# 04 - Avaliação Final do Modelo

## Passos Mágicos - Predição de Risco de Defasagem Escolar

Este notebook apresenta a avaliação final do modelo selecionado.

### Objetivos:
1. Carregar modelo treinado
2. Avaliar métricas finais
3. Analisar importância de features
4. Demonstrar uso da API

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json

from src.models import ModelPredictor, ModelEvaluator
from src.models.predict import RiskPredictor
from src.config import MODELS_DIR, RISCO_LABELS

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Carregar Modelo

In [ ]:
# Carregar modelo
try:
    predictor = ModelPredictor()
    print("Modelo carregado com sucesso!")
    print(f"\nInformações do modelo:")
    print(json.dumps(predictor.get_model_info(), indent=2, default=str))
except FileNotFoundError:
    print("Modelo não encontrado. Execute o notebook 03 primeiro para treinar.")

## 2. Exemplo de Predição

In [ ]:
# Exemplo de predição individual
risk_predictor = RiskPredictor()

# Estudante exemplo - Risco Baixo
resultado_baixo = risk_predictor.predict_risk(
    fase=6,
    idade=13,
    genero="F",
    anos_na_pm=3,
    inde=7.8,
    ian=8.0,
    ida=7.5,
    ieg=8.2,
    iaa=7.9,
    ips=7.5,
    ipp=8.0,
    ipv=7.0,
    pedra="Ametista",
    instituicao_ensino="Escola Estadual",
    bolsista=True
)

print("Estudante com bons indicadores:")
print(f"  Risco: {resultado_baixo['nivel_risco']}")
print(f"  Probabilidade: {resultado_baixo['probabilidade']:.2%}")

In [ ]:
# Estudante exemplo - Risco Alto
resultado_alto = risk_predictor.predict_risk(
    fase=3,
    idade=14,
    genero="M",
    anos_na_pm=1,
    inde=4.5,
    ian=4.0,
    ida=4.2,
    ieg=5.0,
    iaa=4.8,
    ips=4.5,
    ipp=4.0,
    ipv=3.5,
    pedra="Quartzo",
    instituicao_ensino="Escola Municipal",
    bolsista=False
)

print("\nEstudante com indicadores baixos:")
print(f"  Risco: {resultado_alto['nivel_risco']}")
print(f"  Probabilidade: {resultado_alto['probabilidade']:.2%}")

## 3. Teste da API (se estiver rodando)

In [ ]:
# Testar API (se estiver rodando em localhost:8000)
API_URL = "http://localhost:8000"

try:
    # Health check
    response = requests.get(f"{API_URL}/health", timeout=5)
    print("API Status:", response.json())
    
    # Predição
    student_data = {
        "fase": 5,
        "idade": 12,
        "genero": "M",
        "anos_na_pm": 2,
        "inde": 6.5,
        "ian": 7.0,
        "ida": 6.0,
        "ieg": 7.5,
        "iaa": 7.0,
        "ips": 6.5,
        "ipp": 7.0,
        "ipv": 6.0,
        "pedra": "Ágata",
        "instituicao_ensino": "Escola Municipal",
        "bolsista": False
    }
    
    response = requests.post(f"{API_URL}/predict", json=student_data, timeout=10)
    print("\nPredição via API:")
    print(json.dumps(response.json(), indent=2))
    
except requests.exceptions.ConnectionError:
    print("API não está rodando. Execute: make run")
except Exception as e:
    print(f"Erro: {e}")

## 4. Resumo do Projeto

### Modelo Desenvolvido:
- **Objetivo**: Predição de risco de defasagem escolar
- **Target**: 3 classes (BAIXO, MÉDIO, ALTO)
- **Métrica principal**: Recall (priorizar identificação de alunos em risco)

### Pipeline Implementado:
1. **Pré-processamento**: Limpeza, tratamento de missing values
2. **Feature Engineering**: 20+ features derivadas
3. **Seleção de Features**: Top 15 por importância
4. **Modelos Testados**: Logistic Regression, Random Forest, Gradient Boosting
5. **Monitoramento**: Drift detection com Evidently AI

### Entregáveis:
- API FastAPI funcional
- Docker para deploy
- Testes com 80%+ cobertura
- Documentação completa

### Uso em Produção:
```bash
# Local
make run

# Docker
make docker-build
make docker-run
```